In [ ]:
import numpy as np
import rasterio
from scipy.ndimage import binary_dilation

import importlib
ground = importlib.import_module('00_ground_truth_helpers')

In [ ]:
FILE_NUM = '02'

In [ ]:
# Shadow detection
BRIGHTNESS_PERCENTILE = 20
AGG_FRACTION = 0.70    # >70% of 10cm subpixels must be shadow to mark 1m
UPSAMPLE_FACTOR = 10   # 1m -> 10cm

# Shadow-to-tree reassignment
CHM_TREE_HEIGHT = 2.0  # meters
DILATION_METERS = 5    # 1m grid -> 5-pixel dilation

In [ ]:
def pool_brightness_and_threshold(tile_ids, percentile):
    """
    Pool simple-mean RGB brightness at 10cm across tiles and compute the site-wide percentile threshold.

    Inputs:
        tile_ids   : list of tile ID strings
        percentile : percentile of pooled brightness used as the threshold
    Outputs:
        threshold : float, brightness value (DN 0-255) below which pixels are shadow candidates
    """
    pooled = []
    for tid in tile_ids:
        rgb_path, _, _, _ = ground.build_paths(tid)
        with rasterio.open(rgb_path) as src:
            rgb = src.read([1, 2, 3]).astype(np.float32)
            nodata = src.nodata
        if nodata is not None:
            rgb[rgb == nodata] = np.nan
        brightness = np.nanmean(rgb, axis=0)
        v = brightness.ravel()
        v = v[np.isfinite(v)]
        pooled.append(v)
        print(f"tile {tid}: pooled {v.size:,} brightness values")
    pooled = np.concatenate(pooled)
    threshold = float(np.percentile(pooled, percentile))
    print(f"Pooled brightness: n={pooled.size:,}, p{percentile} threshold = {threshold:.2f} DN")
    return threshold

In [ ]:
def detect_shadow_10cm(rgb_path, brightness_threshold):
    """
    Detect shadow pixels at native 10cm from an RGB raster.

    Rule: (brightness < brightness_threshold) AND (B > R).

    Inputs:
        rgb_path             : path to 10cm 3-band RGB GeoTIFF
        brightness_threshold : DN value below which a pixel is brightness-flagged
    Outputs:
        shadow_10cm : 2D boolean array at 10cm resolution
        profile     : source rasterio profile
    """
    with rasterio.open(rgb_path) as src:
        rgb = src.read([1, 2, 3]).astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        rgb[rgb == nodata] = np.nan
    r, g, b = rgb[0], rgb[1], rgb[2]
    brightness = (r + g + b) / 3.0
    shadow_10cm = (brightness < brightness_threshold) & (b > r)
    shadow_10cm &= np.isfinite(brightness) & np.isfinite(r) & np.isfinite(b)
    return shadow_10cm, profile


def aggregate_to_1m(mask_10cm, factor, min_fraction):
    """
    Aggregate a 10cm boolean mask to a 1m boolean mask using a majority rule.

    A 1m pixel is True if strictly more than min_fraction of its 10cm subpixels are True.

    Inputs:
        mask_10cm    : 2D boolean array of shape (H*factor, W*factor)
        factor       : integer downsample factor
        min_fraction : fraction of subpixels required to mark 1m True
    Outputs:
        mask_1m : 2D boolean array of shape (H, W)
    """
    H10, W10 = mask_10cm.shape
    assert H10 % factor == 0 and W10 % factor == 0, f"10cm shape {mask_10cm.shape} not divisible by factor {factor}"
    H1, W1 = H10 // factor, W10 // factor
    reshaped = mask_10cm.reshape(H1, factor, W1, factor)
    frac = reshaped.mean(axis=(1, 3))
    return frac > min_fraction

In [ ]:
def reassign_shadow_with_chm(shadow_1m, chm_path, tree_height, dilation_pixels):
    """
    Split a 1m shadow mask into (a) shadow reassigned to tree and (b) remaining low-confidence shadow,
    based on adjacency to CHM > tree_height.

    Adjacency: shadow pixel falls within dilation_pixels (1m grid, so meters) of any CHM > tree_height pixel.

    Inputs:
        shadow_1m       : 1m boolean shadow mask
        chm_path        : path to 1m CHM GeoTIFF (single band, meters)
        tree_height     : minimum CHM height to be considered a tree (m)
        dilation_pixels : dilation radius in 1m pixels (== meters)
    Outputs:
        shadow_tree    : 1m boolean, shadow reassigned to tree (input to Phase 4)
        shadow_lowconf : 1m boolean, remaining shadow (low-confidence flag)
    """
    with rasterio.open(chm_path) as src:
        chm = src.read(1).astype(np.float32)
        nodata = src.nodata
    if nodata is not None:
        chm[chm == nodata] = np.nan
    tree_footprint = np.where(np.isfinite(chm), chm > tree_height, False)
    size = 2 * dilation_pixels + 1
    struct = np.ones((size, size), dtype=bool)
    tree_zone = binary_dilation(tree_footprint, structure=struct)
    shadow_tree = shadow_1m & tree_zone
    shadow_lowconf = shadow_1m & ~tree_zone
    return shadow_tree, shadow_lowconf


def overwrite_bare_confidence(bare_conf_path, shadow_1m):
    """
    Overwrite the bare confidence raster in place: set confidence to 0 at any pixel where shadow is True
    and confidence is currently > 0 (previously flagged bare).

    Inputs:
        bare_conf_path : path to existing bare_confidence GeoTIFF (float32)
        shadow_1m      : 1m boolean shadow mask (same grid as confidence)
    Outputs:
        n_updated : number of pixels overwritten to 0
    """
    with rasterio.open(bare_conf_path, 'r+') as ds:
        conf = ds.read(1)
        bare = np.isfinite(conf) & (conf > 0)
        overlap = bare & shadow_1m
        conf[overlap] = 0.0
        ds.write(conf, 1)
    return int(overlap.sum())

In [ ]:
def write_boolean_geotiff(mask, reference_profile, out_path):
    """Write a 2D boolean mask as uint8 GeoTIFF using the given rasterio profile."""
    prof = reference_profile.copy()
    prof.update(count=1, dtype='uint8', nodata=255, compress='lzw', height=mask.shape[0], width=mask.shape[1])
    with rasterio.open(out_path, 'w', **prof) as dst:
        dst.write(mask.astype(np.uint8), 1)

In [ ]:
def write_tile_shadow_products(tile_id, brightness_threshold, agg_fraction, upsample_factor, chm_tree_height, dilation_meters):
    """
    Full Phase 3 pipeline for one tile.

    Detects shadow at 10cm, aggregates to 1m, splits into tree-adjacent vs. low-confidence,
    writes 3 mask GeoTIFFs, and overwrites bare_confidence with 0 at shadow-bare overlap.

    Inputs:
        tile_id              : tile ID string
        brightness_threshold : DN threshold for shadow detection
        agg_fraction         : minimum 10cm shadow fraction to mark a 1m pixel as shadow
        upsample_factor      : 1m -> 10cm factor (== 10cm subpixels per 1m pixel per axis)
        chm_tree_height      : minimum CHM height (m) considered a tree for reassignment
        dilation_meters      : dilation radius (m == 1m pixels) for adjacency zone
    Outputs:
        None (writes GeoTIFFs to ground.OUTPUT_DIR)
    """
    rgb_path, _, savi_path, chm_path = ground.build_paths(tile_id)
    bare_conf_path = ground.OUTPUT_DIR / f"{FILE_NUM}_bare_confidence_{tile_id}_{ground.YEAR}.tif"
    print(f"=== Tile {tile_id} ===")

    with rasterio.open(savi_path) as src:
        ref_1m_profile = src.profile.copy()

    print("detecting shadow at 10cm ...")
    shadow_10cm, _ = detect_shadow_10cm(rgb_path, brightness_threshold)
    print(f"shadow fraction at 10cm: {shadow_10cm.mean()*100:.2f}%")

    print(f"aggregating to 1m (>{int(agg_fraction*100)}% majority) ...")
    shadow_1m = aggregate_to_1m(shadow_10cm, upsample_factor, agg_fraction)
    print(f"shadow fraction at 1m : {shadow_1m.mean()*100:.2f}%")

    shadow_path = ground.OUTPUT_DIR / f"{FILE_NUM}_shadow_mask_1m_{tile_id}_{ground.YEAR}.tif"
    write_boolean_geotiff(shadow_1m, ref_1m_profile, shadow_path)
    print(f"wrote {shadow_path.name}")

    print(f"reassigning shadow via CHM > {chm_tree_height}m, {dilation_meters}m dilation ...")
    shadow_tree, shadow_lowconf = reassign_shadow_with_chm(shadow_1m, chm_path, chm_tree_height, dilation_meters)
    print(f"shadow -> tree     : {shadow_tree.sum():,} pixels ({shadow_tree.mean()*100:.2f}%)")
    print(f"shadow -> low-conf : {shadow_lowconf.sum():,} pixels ({shadow_lowconf.mean()*100:.2f}%)")

    tree_shadow_path = ground.OUTPUT_DIR / f"{FILE_NUM}_shadow_tree_mask_{tile_id}_{ground.YEAR}.tif"
    lowconf_path = ground.OUTPUT_DIR / f"{FILE_NUM}_shadow_lowconf_mask_{tile_id}_{ground.YEAR}.tif"
    write_boolean_geotiff(shadow_tree, ref_1m_profile, tree_shadow_path)
    write_boolean_geotiff(shadow_lowconf, ref_1m_profile, lowconf_path)
    print(f"wrote {tree_shadow_path.name}")
    print(f"wrote {lowconf_path.name}")

    print("overwriting bare_confidence with 0 at shadow-bare pixels ...")
    if bare_conf_path.exists():
        n_updated = overwrite_bare_confidence(bare_conf_path, shadow_1m)
        print(f"updated {n_updated:,} pixels in {bare_conf_path.name}")
    else:
        print(f"WARNING: {bare_conf_path.name} not found, skipping overwrite")

In [ ]:
print(f"=== Phase 3: shadow detection ({ground.SITE_ID} {ground.YEAR}) ===")
print(f"Step 1: pooling brightness across {len(ground.TILE_IDS)} tiles ...")
threshold = pool_brightness_and_threshold(ground.TILE_IDS, BRIGHTNESS_PERCENTILE)

In [ ]:
print(f"Step 2: per-tile shadow pipeline (threshold = {threshold:.2f} DN)")
for tid in ground.TILE_IDS:
    write_tile_shadow_products(tid, threshold, AGG_FRACTION, UPSAMPLE_FACTOR, CHM_TREE_HEIGHT, DILATION_METERS)
print("Done.")